# 13 — Model Comparison
**Spacecraft Telemetry Anomaly Detection | Phase 2**

---
**Goal:** Compare Isolation Forest and One-Class SVM side by side across all evaluation dimensions  
and produce a recommendation for Phase 3 (deep learning).

> A good comparison doesn't just look at overall accuracy — it asks:  
> which model is safer for spacecraft operations, and where does each one fail?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, warnings, json
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)

# Load saved results from Section 11 and 12
with open('models/iforest_results.json') as f:
    ifo = json.load(f)
with open('models/ocsvm_results.json') as f:
    svm = json.load(f)

print('Results loaded:')
print(f'  Isolation Forest  — F1: {ifo["f1"]}  AUC: {ifo["auc"]}')
print(f'  One-Class SVM     — F1: {svm["f1"]}  AUC: {svm["auc"]}')

### 13.1 Head-to-Head Metrics Table

In [ ]:
metrics_df = pd.DataFrame({
    'Metric':             ['Precision','Recall','F1 Score','ROC-AUC',
                           'True Positives','False Positives',
                           'False Negatives','True Negatives',
                           'Train Time (s)','Score Time (s)'],
    'Isolation Forest':   [ifo['precision'], ifo['recall'], ifo['f1'], ifo['auc'],
                           ifo['tp'], ifo['fp'], ifo['fn'], ifo['tn'],
                           ifo['train_time'], ifo['score_time']],
    'One-Class SVM':      [svm['precision'], svm['recall'], svm['f1'], svm['auc'],
                           svm['tp'], svm['fp'], svm['fn'], svm['tn'],
                           svm['train_time'], svm['score_time']],
})

# Add which model wins each metric
def winner(row):
    if row['Metric'] in ['False Positives','False Negatives','Train Time (s)','Score Time (s)']:
        return 'IF ✓' if row['Isolation Forest'] < row['One-Class SVM'] else 'OCSVM ✓'
    else:
        return 'IF ✓' if row['Isolation Forest'] > row['One-Class SVM'] else 'OCSVM ✓'

metrics_df['Better Model'] = metrics_df.apply(winner, axis=1)
display(metrics_df)

# KEY NOTE:
# For spacecraft safety: FALSE NEGATIVES are the most dangerous outcome
# A missed anomaly (FN) could mean ignoring a real hardware failure
# --> The model with LOWER False Negatives is preferred for safety-critical use

### 13.2 Metric Bar Chart Comparison

In [ ]:
compare_metrics = ['Precision','Recall','F1 Score','ROC-AUC']
if_vals  = [ifo['precision'], ifo['recall'], ifo['f1'], ifo['auc']]
svm_vals = [svm['precision'], svm['recall'], svm['f1'], svm['auc']]

x   = np.arange(len(compare_metrics))
w   = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, if_vals,  w, label='Isolation Forest',
            color='#1f77b4', alpha=0.85, edgecolor='white')
b2 = ax.bar(x + w/2, svm_vals, w, label='One-Class SVM',
            color='#9467bd', alpha=0.85, edgecolor='white')

ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(compare_metrics, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Isolation Forest vs One-Class SVM — Performance Metrics',
             fontweight='bold')
ax.legend(fontsize=9)
ax.axhline(1.0, color='grey', ls=':', lw=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig('plots_v2/13_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 13.3 ROC Curves — Overlaid

In [ ]:
fpr_if  = np.array(ifo['fpr']);  tpr_if  = np.array(ifo['tpr'])
fpr_svm = np.array(svm['fpr']); tpr_svm = np.array(svm['tpr'])

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_if,  tpr_if,  color='#1f77b4', lw=2.5,
        label=f'Isolation Forest (AUC = {ifo["auc"]:.3f})')
ax.plot(fpr_svm, tpr_svm, color='#9467bd', lw=2.5, ls='--',
        label=f'One-Class SVM    (AUC = {svm["auc"]:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1, alpha=0.5, label='Random baseline (AUC=0.5)')
ax.fill_between(fpr_if, tpr_if, alpha=0.07, color='#1f77b4')
ax.fill_between(fpr_svm, tpr_svm, alpha=0.07, color='#9467bd')

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=11)
ax.set_title('ROC Curves — Both Models Compared\n'
             '(closer to top-left = better)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('plots_v2/13_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 13.4 Confusion Matrices Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrices — Isolation Forest vs One-Class SVM', fontweight='bold')

for ax, res, title, cmap in zip(
    axes,
    [ifo, svm],
    ['Isolation Forest', 'One-Class SVM'],
    ['Blues', 'Purples']
):
    cm = np.array([[res['tn'], res['fp']],
                   [res['fn'], res['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Pred Normal','Pred Anomaly'],
                yticklabels=['True Normal','True Anomaly'],
                ax=ax, linewidths=0.5, cbar=False,
                annot_kws={'size':14, 'weight':'bold'})
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('plots_v2/13_cm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key: Bottom-right = True Positives (caught anomalies)')
print('     Bottom-left  = False Negatives (MISSED anomalies — most dangerous)')

### 13.5 Per Anomaly-Type Recall — Both Models

In [ ]:
# Merge per-type results from both models
if_type_df  = pd.DataFrame(ifo['by_type']).rename(columns={'Recall':'IF Recall','Precision':'IF Precision'})
svm_type_df = pd.DataFrame(svm['by_type']).rename(columns={'Recall':'SVM Recall','Precision':'SVM Precision'})

type_compare = if_type_df[['Anomaly Type','Total','IF Recall','IF Precision']].merge(
    svm_type_df[['Anomaly Type','SVM Recall','SVM Precision']],
    on='Anomaly Type'
)
display(type_compare)

# Visualise per-type recall
atypes   = type_compare['Anomaly Type'].values
x2       = np.arange(len(atypes))

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x2 - 0.2, type_compare['IF Recall'],  0.35, label='Isolation Forest',
            color='#1f77b4', alpha=0.85, edgecolor='white')
b2 = ax.bar(x2 + 0.2, type_compare['SVM Recall'], 0.35, label='One-Class SVM',
            color='#9467bd', alpha=0.85, edgecolor='white')

ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=9)
ax.set_xticks(x2)
ax.set_xticklabels([f'{t}\n(n={n})' for t, n in zip(atypes, type_compare['Total'])],
                   fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Recall')
ax.set_title('Recall by Anomaly Type — Both Models', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots_v2/13_per_type_recall.png', dpi=150, bbox_inches='tight')
plt.show()

# EXPECTED PATTERN TO EXPLAIN:
# Point anomalies → best recall for both models (extreme values isolated easily)
# Contextual anomalies → medium (plausible values, context wrong)
# Collective anomalies → lowest (no single extreme point — pattern-based)
# --> This gap motivates using GRU/TCN autoencoders (Phase 3) which ARE sequence-aware

### 13.6 Speed & Scalability Comparison

In [ ]:
speed_df = pd.DataFrame({
    'Property':          ['Training time','Scoring time','Scales to large data',
                          'Parallelisable','Memory usage'],
    'Isolation Forest':  [f'{ifo["train_time"]:.2f}s', f'{ifo["score_time"]:.3f}s',
                          'Yes — sub-linear', 'Yes (n_jobs=-1)',
                          'Low — tree structures'],
    'One-Class SVM':     [f'{svm["train_time"]:.2f}s', f'{svm["score_time"]:.3f}s',
                          'Poor — quadratic in training size', 'No',
                          'High — kernel matrix + support vectors'],
})
display(speed_df)

# RESULT: Isolation Forest is typically 10-100x faster than OCSVM
# For real-time spacecraft monitoring (telemetry arriving every few seconds),
# Isolation Forest is far more practical for deployment

### 13.7 Final Recommendation & Phase 3 Roadmap

In [ ]:
rec_df = pd.DataFrame({
    'Aspect':          ['Overall performance','Safety (low missed anomalies)',
                        'Operational speed','Point anomalies',
                        'Contextual anomalies','Collective anomalies',
                        'Real-time deployment','Recommended for Phase 2 baseline'],
    'Isolation Forest':['Strong','Good (lower FN)','Excellent',
                        'Excellent','Moderate','Weak',
                        'Yes','✓ PRIMARY CHOICE'],
    'One-Class SVM':   ['Moderate','Moderate','Slow on large data',
                        'Good','Moderate','Weak',
                        'Limited','✓ SECONDARY / OFFLINE'],
})
display(rec_df)

print()
print('─' * 60)
print('  Phase 2 CONCLUSION')
print('─' * 60)
print('  Both classical models perform well on POINT anomalies.')
print('  Both struggle with COLLECTIVE anomalies (slow drifts,   ')
print('  sustained oscillations) because they are not sequence-aware.')
print()
print('  Phase 3 Solution: GRU / TCN Autoencoder')
print('  → Processes sequences of readings, not isolated snapshots')
print('  → Reconstruction error captures drift patterns invisible')
print('    to snapshot-based models')
print('─' * 60)

In [ ]:
# Final summary radar chart
from matplotlib.patches import FancyArrowPatch

categories = ['Precision','Recall','F1','AUC','Speed\n(inv)','Collective\nRecall']
N = len(categories)

# Estimate speed score: normalise train time to [0,1] inverted
max_t = max(ifo['train_time'], svm['train_time'])
if_speed  = 1 - ifo['train_time'] / (max_t + 0.001)
svm_speed = 1 - svm['train_time'] / (max_t + 0.001)

# Collective recall (approximate from per-type data)
def get_collective_recall(data):
    for d in data['by_type']:
        if d['Anomaly Type'] == 'collective':
            return d['Recall']
    return 0.0

if_vals_r  = [ifo['precision'], ifo['recall'], ifo['f1'], ifo['auc'],
               if_speed, get_collective_recall(ifo)]
svm_vals_r = [svm['precision'], svm['recall'], svm['f1'], svm['auc'],
               svm_speed, get_collective_recall(svm)]

angles  = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

if_vals_r  += if_vals_r[:1]
svm_vals_r += svm_vals_r[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, if_vals_r,  color='#1f77b4', lw=2, label='Isolation Forest')
ax.fill(angles, if_vals_r,  color='#1f77b4', alpha=0.15)
ax.plot(angles, svm_vals_r, color='#9467bd', lw=2, ls='--', label='One-Class SVM')
ax.fill(angles, svm_vals_r, color='#9467bd', alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Model Comparison — Radar Chart', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('plots_v2/13_radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()